In [17]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle

# ---------------------------
# Load datasets
# ---------------------------
skills_df = pd.read_excel("datasets/1.xlsx", sheet_name="original")  # Non-academic (skills)
academic_df = pd.read_csv("datasets/2.csv")                          # Academic

# ---------------------------
# Clean and preprocess skills dataset
# ---------------------------
skills_df = skills_df.drop(columns=['Sr.No.', 'Course', 'Student'], errors='ignore')
# Add this right after loading the skills_df
skills_df.columns = skills_df.columns.str.strip().str.replace(' - ', '-')

# Then modify your new_skills_input to use consistent naming:
# Then modify your new_skills_input to use consistent naming:
new_skills_input = {
    'Linguistic': 80,
    'Musical': 65,
    'Bodily': 70,
    'Logical-Mathematical': 75,  # Now matches processed column names
    'Spatial-Visualization': 68,
    'Interpersonal': 85,
    'Intrapersonal': 90,
    'Naturalist': 70,
    'P1': 2, 'P2': 1, 'P3': 2, 'P4': 1, 'P5': 2, 'P6': 1, 'P7': 0, 'P8': 1
}
skills_df = skills_df.dropna()

# Map POOR/AVG/BEST → 0/1/2
performance_map = {"POOR": 0, "AVG": 1, "BEST": 2}
for col in ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8']:
    skills_df[col] = skills_df[col].map(performance_map)

# Encode career labels
skills_label = LabelEncoder()
skills_df['career'] = skills_label.fit_transform(skills_df['Job profession'])

# Prepare features and target
X_skills = skills_df.select_dtypes(include=[np.number]).drop(columns=['career'], errors='ignore')
y_skills = skills_df['career']

# Train-test split for skills model
X_sk_train, X_sk_test, y_sk_train, y_sk_test = train_test_split(
    X_skills, y_skills, test_size=0.3, random_state=42
)
rf_skills = RandomForestClassifier(n_estimators=100, random_state=42)
rf_skills.fit(X_sk_train, y_sk_train)

# Save skills model
with open("rf_skills_model.pkl", "wb") as f:
    pickle.dump((rf_skills, skills_label), f)

# ---------------------------
# Clean and preprocess academic dataset
# ---------------------------
# Drop personal identifiers
academic_df = academic_df.drop(columns=['id', 'first_name', 'last_name', 'email'], errors='ignore')
academic_df = academic_df.dropna()

# Convert boolean columns to int
bool_cols = ['part_time_job', 'extracurricular_activities']
for col in bool_cols:
    academic_df[col] = academic_df[col].astype(int)

# Encode gender
academic_df['gender'] = academic_df['gender'].map({'male': 0, 'female': 1})

# Encode career aspiration
acad_label = LabelEncoder()
academic_df['career'] = acad_label.fit_transform(academic_df['career_aspiration'])

# Prepare features and target
X_academic = academic_df.select_dtypes(include=[np.number]).drop(
    columns=['career', 'career_aspiration'], errors='ignore'
)
y_academic = academic_df['career']

# Train-test split for academic model
X_ac_train, X_ac_test, y_ac_train, y_ac_test = train_test_split(
    X_academic, y_academic, test_size=0.3, random_state=42
)
rf_academic = RandomForestClassifier(n_estimators=100, random_state=42)
rf_academic.fit(X_ac_train, y_ac_train)

# Save academic model
with open("rf_academic_model.pkl", "wb") as f:
    pickle.dump((rf_academic, acad_label), f)

# ---------------------------
# Evaluate models
# ---------------------------
print("🎯 Skills Model Accuracy:", accuracy_score(y_sk_test, rf_skills.predict(X_sk_test)))
print("🎓 Academic Model Accuracy:", accuracy_score(y_ac_test, rf_academic.predict(X_ac_test)))

# ---------------------------
# Prediction function
# ---------------------------
def predict_career(skills_input=None, academic_input=None):
    if skills_input is not None:
        skills_df = pd.DataFrame([skills_input])
        career_skills = skills_label.inverse_transform(rf_skills.predict(skills_df))[0]
        print(f"🎯 Predicted career based on skills: {career_skills}")
    
    if academic_input is not None:
        academic_df = pd.DataFrame([academic_input])
        career_academic = acad_label.inverse_transform(rf_academic.predict(academic_df))[0]
        print(f"🎓 Predicted career based on academics: {career_academic}")

# Example usage
new_skills_input = {
    'Linguistic': 80,
    'Musical': 80,
    'Bodily': 60,
    'Logical-Mathematical': 60,
    'Spatial-Visualization': 50,
    'Interpersonal': 85,
    'Intrapersonal': 90,
    'Naturalist': 70,
    'P1': 2, 'P2': 2, 'P3': 1, 'P4': 1, 'P5': 2, 'P6': 2, 'P7': 1, 'P8': 1
}

new_academic_input = {
    'gender': 1,  # female
    'part_time_job': 0,
    'absence_days': 2,
    'extracurricular_activities': 1,
    'weekly_self_study_hours': 10,
    'math_score': 85,
    'history_score': 72,
    'physics_score': 78,
    'chemistry_score': 76,
    'biology_score': 88,
    'english_score': 91,
    'geography_score': 80
}

predict_career(skills_input=new_skills_input, academic_input=new_academic_input)

🎯 Skills Model Accuracy: 0.9721706864564007
🎓 Academic Model Accuracy: 0.955
🎯 Predicted career based on skills: Manager
🎓 Predicted career based on academics: Unknown


🎯 Skills Model Accuracy: 0.9721706864564007
🎓 Academic Model Accuracy: 0.955


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Logical-Mathematical
Feature names seen at fit time, yet now missing:
- Logical - Mathematical
